In [2]:
import tensorflow as tf 
import numpy as np
from tensorflow.keras.layers import LSTM, Embedding, Dense 
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer 
import sklearn as sk 
from tensorflow.keras.utils import to_categorical
import pandas as pd 
from sklearn.preprocessing import LabelEncoder

I0000 00:00:1779207088.908140   12195 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779207089.135358   12195 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [11]:
dat = pd.read_csv("sentiment_analysis.csv")
dat = dat.drop(dat[['Year']],  axis = 1)
dat = dat.drop(dat[['Month']], axis = 1)
dat = dat.drop(dat[['Day']], axis = 1)
dat = dat.drop(dat[['Time of Tweet']], axis = 1)
dat = dat.drop(dat[['Platform']], axis = 1)

x = dat['text']
y = dat['sentiment']

#encoding output variable 
le = LabelEncoder()
y = le.fit_transform(y)


#turning input variable(sentences) into tokens
token = Tokenizer(num_words = 10000)
token.fit_on_texts(x)
sequence = token.texts_to_sequences(x)
# padding to ensure that all the elements of the 2D list is of same length
padded_tr_array = tf.keras.utils.pad_sequences(
    sequence,
    maxlen = 26,
    dtype='int32',
    padding='pre',
    truncating='pre',
    value=0.0
)

print(f"input label: {padded_tr_array.shape} output label : {y.shape}")

input label: (499, 26) output label : (499,)


In [5]:
#training the model

model = Sequential([
    Embedding(input_dim = 10000, output_dim = 64),
    LSTM(128, return_sequences = True),#when stacking LSTM, the second LSTM expects a sequence, so put return_sequence = True
    LSTM(64),
    Dense(32, activation = 'relu'),
    Dense(16, activation = 'relu'),
    Dense(3, activation = 'softmax')
])
model.compile(loss='sparse_categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])
mod_history = model.fit(padded_tr_array, y, batch_size = 4, epochs = 10)

Epoch 1/10


I0000 00:00:1779207115.802177   12195 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4168 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1779207118.314601   17354 cuda_dnn.cc:461] Loaded cuDNN version 91700


125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3667 - loss: 1.0971
Epoch 2/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.4709 - loss: 1.0308
Epoch 3/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6713 - loss: 0.6739
Epoch 4/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8818 - loss: 0.3250
Epoch 5/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9679 - loss: 0.1069
Epoch 6/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9880 - loss: 0.0376
Epoch 7/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9880 - loss: 0.0480
Epoch 8/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9960 - loss: 0.0252
Epoch 9/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9960 - loss: 0.0139
Epoch 10/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9960 - loss: 0.0107


In [10]:
# output prediction for random input 

text = input("Enter user comment here: ")

sequence = token.texts_to_sequences([text])

padded_tr_array1 = tf.keras.utils.pad_sequences(
    sequence,
    maxlen = 26,
    dtype='int32',
    padding='pre',
    truncating='pre',
    value=0.0
)
prediction = model.predict(padded_tr_array1)
predicted_class = np.argmax(prediction)

print(predicted_class)
label = le.inverse_transform([predicted_class])

print(label)

Enter user comment here:  im angry 


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
0
['negative']
